Этап 2. Загрузка и первичный осмотр данных

**Источник:** Natural Resources Canada, Fuel Consumption Ratings (open.canada.ca), лицензия Open Government Licence Canada.
**Файл:** `my2015-2024-fuel-consumption-ratings.csv`: основной файл для обучения и тестирования. Файлы 1995-2014, 2025 и 2026 не смешиваем с основным.

**Что делаем:** читаем CSV в таблицу `df` через `pd.read_csv`.
- Путь относительный (`../data/raw/...`), чтобы ноутбук запускался у всех участников команды.
- `na_values=["n/a"]`: текст `n/a` читается как настоящий пропуск (NaN), а не как строка.

In [101]:
import pandas as pd

# Путь относительно папки notebooks/
path = "../data/raw/my2015-2024-fuel-consumption-ratings.csv"

# na_values: чтобы текст "n/a" читался как настоящий пропуск (NaN), а не как строка
df = pd.read_csv(path, na_values=["n/a"])

### Размер данных

`df.shape` показывает число строк и столбцов.

**Результат:** (10060, 15), то есть 10060 автомобилей и 15 столбцов.
**Вывод:** для Linear Regression, Decision Tree и KNN этого достаточно, модели обучаются быстро.

In [102]:
print(df.shape)

(10060, 15)


### Типы данных и пропуски

`df.info()` показывает тип каждого столбца и сколько в нём непустых значений.

**Результат:**
- Типы: 6 столбцов float64, 4 int64, 5 текстовых.
- Пропусков нет везде, кроме двух столбцов: `CO2 rating` (8932 из 10060 заполнены) и `Smog rating` (7826 из 10060).

**Вывод:** оба столбца с пропусками мы всё равно не берём в признаки из-за утечки данных (см. ниже), поэтому заполнять пропуски не нужно.

In [103]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 10060 entries, 0 to 10059
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Model year            10060 non-null  int64  
 1   Make                  10060 non-null  str    
 2   Model                 10060 non-null  str    
 3   Vehicle class         10060 non-null  str    
 4   Engine size (L)       10060 non-null  float64
 5   Cylinders             10060 non-null  int64  
 6   Transmission          10060 non-null  str    
 7   Fuel type             10060 non-null  str    
 8   City (L/100 km)       10060 non-null  float64
 9   Highway (L/100 km)    10060 non-null  float64
 10  Combined (L/100 km)   10060 non-null  float64
 11  Combined (mpg)        10060 non-null  int64  
 12  CO2 emissions (g/km)  10060 non-null  int64  
 13  CO2 rating            8932 non-null   float64
 14  Smog rating           7826 non-null   float64
dtypes: float64(6), int64(4), str(5

### Первые строки

`df.head()` показывает первые 5 строк, чтобы увидеть, как выглядят данные.

**Что видно:**
- Данные начинаются с 2015 года, значит, загружен правильный файл.
- `Vehicle class` записан в разных форматах (например, `Compact` и `Sport utility vehicle: Small`): категории нужно привести к единому виду.
- `Transmission` состоит из типа и числа передач (`AS5`, `M6`, `AV7`): из неё можно сделать два признака.
- `Fuel type` закодирован буквами.
- В названии модели встречаются гибриды (например, `ILX Hybrid`).

In [104]:
print(df.head())

   Model year   Make       Model                 Vehicle class  \
0        2015  Acura         ILX                       Compact   
1        2015  Acura         ILX                       Compact   
2        2015  Acura  ILX Hybrid                       Compact   
3        2015  Acura  MDX SH-AWD  Sport utility vehicle: Small   
4        2015  Acura     RDX AWD  Sport utility vehicle: Small   

   Engine size (L)  Cylinders Transmission Fuel type  City (L/100 km)  \
0              2.0          4          AS5         Z              9.7   
1              2.4          4           M6         Z             10.8   
2              1.5          4          AV7         Z              6.0   
3              3.5          6          AS6         Z             12.7   
4              3.5          6          AS6         Z             12.1   

   Highway (L/100 km)  Combined (L/100 km)  Combined (mpg)  \
0                 6.7                  8.3              34   
1                 7.4                  9

### Список столбцов

`df.columns.tolist()` выводит точные названия столбцов, чтобы обращаться к ним в коде без опечаток.

**Кандидаты в признаки:** `Model year`, `Make`, `Model`, `Vehicle class`, `Engine size (L)`, `Cylinders`, `Transmission`, `Fuel type`.

**Target:** `Combined (L/100 km)`.

**Исключаем из признаков (data leakage):** `City (L/100 km)`, `Highway (L/100 km)`, `Combined (mpg)`, `CO2 emissions (g/km)`, `CO2 rating`, `Smog rating`. Они напрямую выражают target или считаются из него, и модель получила бы ответ вместо того, чтобы его предсказывать.

In [105]:
print(df.columns.tolist())

['Model year', 'Make', 'Model', 'Vehicle class', 'Engine size (L)', 'Cylinders', 'Transmission', 'Fuel type', 'City (L/100 km)', 'Highway (L/100 km)', 'Combined (L/100 km)', 'Combined (mpg)', 'CO2 emissions (g/km)', 'CO2 rating', 'Smog rating']


### Диапазон лет, типы топлива и статистика

В ячейке три проверки:
1. Минимум и максимум года. **Результат:** 2015 и 2024, файл верный.
2. `value_counts()` по `Fuel type`. **Результат:** X: 4756, Z: 4739, E: 328, D: 236, N: 1. Категории очень разной частоты, а `N` встречается всего один раз (решение по нему примем при очистке). Расшифровку букв сверим с документацией источника.
3. `describe()`: минимум, максимум, среднее по числовым столбцам.

**Что видно в `describe()`:**
- Target `Combined (L/100 km)`: от 4.0 до 26.1, среднее около 11.0, std около 2.8. Разброс достаточный, чтобы было что предсказывать.
- Объём двигателя 0.9-8.4 л, цилиндры 3-16: значения правдоподобны, явных ошибок не видно.

**Ограничение датасета (limitation):** в данных нет веса и мощности автомобиля.

In [106]:
print(df["Model year"].min(), df["Model year"].max())
print(df["Fuel type"].value_counts())
df.describe()

2015 2024
Fuel type
X    4756
Z    4739
E     328
D     236
N       1
Name: count, dtype: int64


,Model year,Engine size (L),Cylinders,City (L/100 km),Highway (L/100 km),Combined (L/100 km),Combined (mpg),CO2 emissions (g/km),CO2 rating,Smog rating
count,10060.000000,10060.000000,10060.00000,10060.000000,10060.000000,10060.000000,10060.000000,10060.000000,8932.000000,7826.000000
mean,2019.242048,3.144901,5.61839,12.483052,9.142803,10.979712,27.436282,253.711332,4.617443,4.816509
std,2.834109,1.340010,1.87591,3.404483,2.178649,2.808468,7.316270,60.474917,1.574763,1.783793
min,2015.000000,0.900000,3.00000,4.000000,3.900000,4.000000,11.000000,94.000000,1.000000,1.000000
25%,2017.000000,2.000000,4.00000,10.100000,7.600000,9.000000,22.000000,210.000000,4.000000,3.000000
50%,2019.000000,3.000000,6.00000,12.100000,8.900000,10.700000,26.000000,250.000000,5.000000,5.000000
75%,2022.000000,3.700000,6.00000,14.500000,10.400000,12.700000,31.000000,292.000000,5.000000,6.000000
max,2024.000000,8.400000,16.00000,30.700000,20.900000,26.100000,71.000000,608.000000,10.000000,8.000000


# Этап 3. Очистка данных
Проверяем: дубликаты, пропуски, диапазоны, категории, подозрительные признаки (leakage).
Каждое решение записываем с причиной.

## 3.1 Дубликаты

**Что проверяем:** есть ли в таблице полностью одинаковые строки (совпадают все 15 столбцов).
**Зачем:** повторяющиеся строки учили бы модель на одних и тех же данных несколько раз и делали бы оценку качества менее честной.
**Как:** `df.duplicated().sum()` считает повторные копии строк.

In [107]:
# Сколько полностью одинаковых строк (вторая и последующие копии)
print("Дубликатов:", df.duplicated().sum())
# Показываем все копии, чтобы понять, ошибка это или реальные машины
df[df.duplicated(keep=False)].sort_values(["Make", "Model", "Model year"]).head(10)

Дубликатов: 0


,Model year,Make,Model,Vehicle class,Engine size (L),Cylinders,Transmission,Fuel type,City (L/100 km),Highway (L/100 km),Combined (L/100 km),Combined (mpg),CO2 emissions (g/km),CO2 rating,Smog rating


**Результат:** найдено 0 дубликатов (`df.duplicated().sum()` = 0). Вторая часть ячейки вернула пустую таблицу, потому что копий строк нет.

**Решение:** ничего не удаляем.

**Причина:** полностью одинаковых строк нет, поэтому модель не увидит одни и те же данные несколько раз, и оценка качества не будет искажена повторами.

## 3.2 Пропуски в CO2 rating и Smog rating

**Что делаем:** считаем для каждого года, сколько рейтингов пропущено.

**Зачем:** проверяем гипотезу, что пропуски связаны с годом (рейтинг не публиковался), а не разбросаны случайно.
Это нужно, чтобы понимать причину пропусков, даже если мы их не заполняем.

In [108]:
# Для каждого года: сколько машин всего и сколько пропусков в рейтингах
by_year = df.groupby("Model year").agg(
    rows=("Model year", "size"),
    co2_rating_missing=("CO2 rating", lambda s: s.isna().sum()),
    smog_rating_missing=("Smog rating", lambda s: s.isna().sum()),
)
print(by_year)

            rows  co2_rating_missing  smog_rating_missing
Model year                                               
2015        1128                1128                 1128
2016        1106                   0                 1106
2017        1058                   0                    0
2018        1083                   0                    0
2019        1056                   0                    0
2020         975                   0                    0
2021         970                   0                    0
2022         976                   0                    0
2023         919                   0                    0
2024         789                   0                    0


**Результат:** пропуски в рейтингах связаны с годом, а не разбросаны случайно:
- `CO2 rating` пропущен во всех 1128 строках 2015 года, начиная с 2016 года пропусков нет;
- `Smog rating` пропущен во всех строках 2015 и 2016 годов (1128 + 1106 = 2234), начиная с 2017 года пропусков нет.

Гипотеза подтвердилась: для этих лет значений в файле нет.

**Решение:** пропуски не заполняем и строки с ними не удаляем.

**Причина:** столбцы `CO2 rating` и `Smog rating` уйдут из признаков как leakage (они почти напрямую выражают target), поэтому заполнять их нет смысла. Строки 2015 и 2016 годов при этом сохраняются: мы убираем только столбцы, а не машины, и не теряем 2234 наблюдения.

## 3.3 Категориальные столбцы

**Что делаем:** смотрим, какие значения встречаются в `Vehicle class`, `Transmission` и `Fuel type`, и проверяем текстовые столбцы на лишние пробелы и разный регистр.

**Зачем:** модели работают с числами, поэтому текстовые категории позже придётся кодировать. Если одна и та же категория записана по-разному (например `Compact` и `compact `), кодировщик посчитает их разными, и признак станет менее информативным.

**Как:** `value_counts()` показывает все значения и их частоту. Проверка пробелов и регистра сравнивает число уникальных значений до и после приведения к единому виду.

In [109]:
# Классы автомобилей: сколько машин в каждом
df["Vehicle class"].value_counts()
# Трансмиссия: уникальные коды и их частота
df["Transmission"].value_counts()

Transmission
AS8     2017
AS6     1280
M6      1033
A8       793
AM7      697
A9       642
A6       633
AS10     508
AV       410
A10      279
AS7      265
AM8      261
AM6      162
AS9      161
M5       159
AV7      148
AV6      141
M7       140
AV8      120
A5        54
A7        46
AV10      39
AS5       24
A4        20
AV1       19
AM9        6
AM5        2
AS4        1
Name: count, dtype: int64

In [110]:
# Классы автомобилей и сколько машин в каждом
print(df["Vehicle class"].value_counts())

Vehicle class
Sport utility vehicle: Small       1868
Mid-size                           1393
Sport utility vehicle: Standard    1231
Compact                            1135
Pickup truck: Standard              878
Subcompact                          850
Full-size                           779
Two-seater                          639
Minicompact                         460
Station wagon: Small                282
Pickup truck: Small                 220
Special purpose vehicle             103
Minivan                              95
Station wagon: Mid-size              82
Van: Passenger                       45
Name: count, dtype: int64


In [111]:
# Если "уникальных" и "после приведения" совпадают, проблем с пробелами и регистром нет
for col in ["Make", "Model", "Vehicle class", "Transmission", "Fuel type"]:
    n_space = (df[col] != df[col].str.strip()).sum()
    n_unique = df[col].nunique()
    n_unique_norm = df[col].str.strip().str.lower().nunique()
    print(col, "| с лишними пробелами:", n_space,
          "| уникальных:", n_unique, "| после приведения:", n_unique_norm)

Make | с лишними пробелами: 0 | уникальных: 42 | после приведения: 42
Model | с лишними пробелами: 21 | уникальных: 2080 | после приведения: 2058
Vehicle class | с лишними пробелами: 0 | уникальных: 15 | после приведения: 15
Transmission | с лишними пробелами: 0 | уникальных: 28 | после приведения: 28
Fuel type | с лишними пробелами: 0 | уникальных: 5 | после приведения: 5


## 3.4 Fuel type = N

**Что делаем:** смотрим на единственную строку с типом топлива `N`.

**Зачем:** одна строка на 10060 ничему не научит модель. При разбиении на train и test она может попасть только в одну часть, и тогда кодировщик встретит неизвестную категорию. Нужно решить, что с ней делать.

In [112]:
# Единственная строка с Fuel type = N
print(df[df["Fuel type"] == "N"])

# Удаляем её и проверяем размер
print("До:", df.shape)
df = df[df["Fuel type"] != "N"].reset_index(drop=True)
print("После:", df.shape)
print(df["Fuel type"].value_counts())

      Model year       Make             Model Vehicle class  Engine size (L)  \
1364        2016  Chevrolet  Impala Dual Fuel      Mid-size              3.6   

      Cylinders Transmission Fuel type  City (L/100 km)  Highway (L/100 km)  \
1364          6          AS6         N             15.2                 9.5   

      Combined (L/100 km)  Combined (mpg)  CO2 emissions (g/km)  CO2 rating  \
1364                 12.7              22                   213         6.0   

      Smog rating  
1364          NaN  
До: (10060, 15)
После: (10059, 15)
Fuel type
X    4756
Z    4739
E     328
D     236
Name: count, dtype: int64


**Результат:** единственная строка с `N` это Chevrolet Impala Dual Fuel 2016 (3.6 л, 6 цилиндров, AS6). После удаления в таблице 10059 строк и 4 типа топлива: X, Z, E, D.

**Решение:** удаляем эту строку.

**Причина:** одна строка не позволит модели выучить закономерность. Объединять `N` с другой категорией нельзя, это другой вид топлива. Потеря 0,01% данных.

## 3.5 Названия моделей (Model)

**Что делаем:** ищем в `Model` лишние пробелы и разный регистр у одинаковых названий.

**Зачем:** пару `Make` + `Model` мы планируем использовать для group split. Если одна модель записана по-разному, она попадёт в разные группы, и почти одинаковые машины окажутся и в train, и в test (leakage).


In [113]:
# Строки с пробелами по краям (repr показывает кавычки, пробелы видны)
bad = df[df["Model"] != df["Model"].str.strip()]
print("Строк с пробелами:", len(bad))
bad["Model"].map(repr).head(10)

Строк с пробелами: 21


1711               'XF AWD '
4010                 'CX-5 '
5461           'A6 quattro '
5462           'A6 quattro '
5463           'A6 allroad '
5465                  'A8L '
5466                  'A8L '
6115         'A 220 4MATIC '
6127    'AMG CLA 35 4MATIC '
6128    'AMG CLA 45 4MATIC '
Name: Model, dtype: str

In [114]:
# Уникальных названий: после strip и после strip + lower
s = df["Model"].str.strip()
print("после strip:", s.nunique(), "| после strip + lower:", s.str.lower().nunique())

после strip: 2070 | после strip + lower: 2058


In [115]:
# Названия, совпадающие без учёта регистра, но записанные по-разному
low = s.str.lower()
n_variants = s.groupby(low).nunique()
for key in n_variants[n_variants > 1].index:
    print(sorted(s[low == key].unique()))

['ALPINA B8 Gran Coupe', 'Alpina B8 Gran Coupe']
['ALPINA XB7', 'Alpina XB7']
['GranTurismo', 'Granturismo']
['GranTurismo Convertible', 'Granturismo Convertible']
['Huracan EVO Coupe', 'Huracan evo Coupe']
['Huracan EVO Coupe AWD', 'Huracan evo Coupe AWD']
['Huracan EVO Spyder', 'Huracan evo Spyder']
['Huracan EVO Spyder AWD', 'Huracan evo Spyder AWD']
['LaCrosse', 'Lacrosse']
['LaCrosse AWD', 'Lacrosse AWD']
['LaCrosse eAssist', 'Lacrosse eAssist']
['Tacoma 4WD D-Cab TRD Off-Road/PRO', 'Tacoma 4WD D-Cab TRD Off-Road/Pro']


In [116]:
# Сколько уникальных пар Make + Model до приведения регистра
pairs_before = df[["Make", "Model"]].drop_duplicates().shape[0]

# Приводим к нижнему регистру (пробелы уже убраны)
df["Model"] = df["Model"].str.lower()

# Проверка: пар должно стать ровно на 12 меньше
pairs_after = df[["Make", "Model"]].drop_duplicates().shape[0]
print("Пар Make+Model: до", pairs_before, "| после", pairs_after,
      "| убрано", pairs_before - pairs_after)
print("Уникальных Model:", df["Model"].nunique())

Пар Make+Model: до 2081 | после 2069 | убрано 12
Уникальных Model: 2068


**Результат:**
- В `Model` у 21 строки были лишние пробелы по краям. После `strip` число уникальных названий снизилось с 2080 до 2070.
- 12 названий были записаны с разным регистром у одной и той же модели (например `LaCrosse` и `Lacrosse`, `Huracan EVO Coupe` и `Huracan evo Coupe`). После приведения к нижнему регистру уникальных названий 2058, а пар `Make` + `Model` стало 2059 вместо 2071 (убрано ровно 12).
- Пар (2059) на одну больше, чем названий (2058): одно и то же название встречается у двух разных марок.

**Решение:** убрали пробелы по краям и привели `Model` к нижнему регистру.
**Причина:** это одни и те же автомобили, записанные по-разному. Для group split они попали бы в разные группы, и почти одинаковые машины оказались бы одновременно в train и в test (leakage). Группировать нужно по паре `Make` + `Model`, а не по одному `Model`: иначе две разные машины с одинаковым названием у разных марок склеились бы. Сам `Model` как признак не используем, названий слишком много.

## 3.6 Диапазоны и выбросы
**Что делаем:** смотрим минимум и максимум ключевых числовых столбцов.
**Зачем:** найти физически невозможные значения (нулевой двигатель, отрицательный расход). Большие и маленькие, но реальные значения удалять нельзя.

In [117]:
# Диапазоны ключевых числовых столбцов
df[["Engine size (L)", "Cylinders", "Combined (L/100 km)"]].describe()

,Engine size (L),Cylinders,Combined (L/100 km)
count,10059.000000,10059.000000,10059.000000
mean,3.144855,5.618352,10.979541
std,1.340069,1.875999,2.808556
min,0.900000,3.000000,4.000000
25%,2.000000,4.000000,9.000000
50%,3.000000,6.000000,10.700000
75%,3.700000,6.000000,12.700000
max,8.400000,16.000000,26.100000


**Результат:** по 10059 строкам: `Engine size` 0.9-8.4 л, `Cylinders` 3-16, `Combined (L/100 km)` 4.0-26.1 (среднее около 11, std около 2.8). Нулей и отрицательных значений нет.
**Решение:** выбросы не удаляем.
**Причина:** высокий расход у мощных машин и низкий у гибридов это реальные автомобили, а не ошибки. Удалять стоит только физически невозможные значения, а таких нет.

## 3.7 Столбцы с утечкой (leakage)
**Что делаем:** фиксируем список столбцов, которые нельзя использовать как признаки.

**Зачем:** большая часть этих столбцов почти напрямую выражает target `Combined (L/100 km)`: `City` и `Highway` это его составные части, `Combined (mpg)` это тот же расход в других единицах, `CO2 emissions` рассчитывается из расхода по коэффициенту для типа топлива, `CO2 rating` выводится из CO2. Модель получила бы ответ в подсказке и показала бы красивые, но ложные результаты. `Smog rating` с расходом напрямую не связан, но мы его тоже убираем: это официальная оценка, а не характеристика автомобиля, и она пропущена за 2015-2016 годы.

**Как:** сохраняем список в `leak_cols`. Из `df` пока не удаляем, они нужны для EDA (корреляционная матрица покажет проблему). Удалим при сборке признаков.

In [118]:
leak_cols = [
    "City (L/100 km)", "Highway (L/100 km)", "Combined (mpg)",
    "CO2 emissions (g/km)", "CO2 rating", "Smog rating",
]
print(leak_cols)

['City (L/100 km)', 'Highway (L/100 km)', 'Combined (mpg)', 'CO2 emissions (g/km)', 'CO2 rating', 'Smog rating']


**Результат:** в `leak_cols` зафиксированы 6 столбцов: `City (L/100 km)`, `Highway (L/100 km)`, `Combined (mpg)`, `CO2 emissions (g/km)`, `CO2 rating`, `Smog rating`.

**Решение:** эти столбцы не станут признаками. В `df` они пока остаются.

**Причина:** пять из шести столбцов выражают target напрямую или через пересчёт, шестой (`Smog rating`) убираем по правилу проекта и из-за пропусков. Оставляем их до EDA, чтобы показать на корреляционной матрице, почему они дали бы утечку. Из таблицы признаков удалим при сборке признаков (этап 5), до split и до любого `fit`.

## 3.8 Сохранение очищенных данных
**Что делаем:** сохраняем очищенную таблицу в `data/processed/fuel_clean.csv` и читаем её обратно.

**Зачем:** получаем контрольную точку: файл можно открыть отдельно и убедиться, что очистка ничего не потеряла. Проверка `shape` до и после чтения показывает, что строки и столбцы на месте. `index=False` не даёт записать в файл лишний столбец с номерами строк.

In [119]:
# Сохраняем очищенную таблицу и проверяем, что она читается обратно
out_path = "../data/processed/fuel_clean.csv"
df.to_csv(out_path, index=False)

check = pd.read_csv(out_path)
print("Сохранено:", df.shape, "| прочитано:", check.shape)

Сохранено: (10059, 15) | прочитано: (10059, 15)


**Результат:** сохранено `(10059, 15)`, прочитано обратно `(10059, 15)`. Размеры совпадают.

## Итог этапа 3
- **Строк:** было 10060, стало 10059 (удалена 1 строка с `Fuel type = N`).
- **Дубликаты:** нет.
- **Пропуски:** только `CO2 rating` (2015) и `Smog rating` (2015-2016). Не заполняем, оба столбца в `leak_cols`.
- **Категории:** `Vehicle class`, `Transmission`, `Make` не меняем. В `Model` убрали пробелы у 21 строки и привели регистр (12 вариантов написания).
- **Выбросы:** не удаляли, невозможных значений нет.
- **Leakage:** в `leak_cols` зафиксированы 6 столбцов, которые не станут признаками.
- **Заметки на следующие этапы:** разобрать `Transmission` на тип и число передач (у `AV` без числа 410 строк); для group split использовать пару `Make` + `Model`; кодировщик настроить на неизвестные категории из-за редких классов `Vehicle class`.
- **Данные:** очищенная таблица сохранена в `data/processed/fuel_clean.csv`.